# Nodal corrections: UTide vs pytides vs pyTMD

Every harmonic tidal analysis multiplies each constituent by a slowly varying
**nodal factor** $f(t)$ and shifts it by a **nodal phase** $u(t)$, which track the
18.61-year precession of the lunar node:

$$h(t) = Z_0 + \sum_k \underbrace{f_k(t)}_{\text{amplitude}} A_k
         \cos\!\big[\omega_k t + \underbrace{(V_k + u_k(t))}_{\text{argument}} - g_k\big]$$

Three widely used Python packages compute $(f, u)$ from different source tables:

| package | source of $(f, u)$ |
|---|---|
| **UTide** | Foreman (1977) satellite-modulation table (`ut_constants.npz`) |
| **pytides** | Schureman (1958) closed-form formulas |
| **pyTMD** | Doodson/Schureman/Foreman hybrid (`corrections='OTIS'`) |

 * **Part 1** compares the tables themselves over a full nodal cycle.
 * **Part 2** asks the question that actually matters: feed the same six-year
tide-gauge record to UTide and pytides and see how much of the table-level
disagreement survives into fitted amplitudes and into the detided surge.

> **TL;DR** Most of the disagreement is harmless:
> on a six-year record the two packages agree to a fraction of a millimetre on
> every major constituent. It concentrates almost entirely in **Mf** and **S1**,
> for two different reasons, and how much it costs you depends on the station.


## Part 1 — the correction tables, over one nodal cycle

Each backend is wrapped in a function returning $(f, u)$ on a shared
`DatetimeIndex`, then stacked into one tidy `xr.Dataset` with dims
`(package, time, constituent)` so the comparison is a single `.sel()` away.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import hvplot.xarray  # noqa: F401
import holoviews as hv

hv.extension('bokeh')

import pyTMD, timescale
from utide.harmonics import FUV
from utide._ut_constants import ut_constants
from utide._time_conversion import _normalize_time
from pytides2 import constituent as pytidesc
from pytides2.astro import astro as py_astro

LAT = -45.0
UT_NAMES = [n.strip() for n in ut_constants.const.name]

# const: (UTide name, pytides object, pyTMD name)
CONS = {
    "Mf":   ("MF",   pytidesc._Mf,   "mf"),
    "Mm":   ("MM",   pytidesc._Mm,   "mm"),
    "MSf":  ("MSF",  pytidesc._MSF,  "msf"),
    "S1":   ("S1",   pytidesc._S1,   "s1"),
    "R2":   ("R2",   pytidesc._R2,   "r2"),
    "M1 (or NO1)":   ("NO1",  pytidesc._M1,   "m1"),
    "O1":   ("O1",   pytidesc._O1,   "o1"),
    "K1":   ("K1",   pytidesc._K1,   "k1"),
    "M2":   ("M2",   pytidesc._M2,   "m2"),
    "S2":   ("S2",   pytidesc._S2,   "s2"),
    "K2":   ("K2",   pytidesc._K2,   "k2"),
    "L2":   ("L2",   pytidesc._L2,   "l2"),
    "2N2":  ("2N2",  pytidesc._2N2,  "2n2"),
    "J1":   ("J1",   pytidesc._J1,   "j1"),
    "OO1":  ("OO1",  pytidesc._OO1,  "oo1"),
    "P1":   ("P1",   pytidesc._P1,   "p1"),
}

wrap180 = lambda x: (np.asarray(x, float) + 180) % 360 - 180

In [ ]:
def fu_utide(t, names, lat=LAT):
    tn = _normalize_time(t.values.astype("datetime64[s]"))
    lind = np.array([UT_NAMES.index(n) for n in names])
    F, U, _ = FUV(tn, tn[0], lind, lat, [0, 0, 0, 0])
    return F, U * 360.0


def fu_pytides(t, objs):
    a = [py_astro(x.to_pydatetime()) for x in t]
    f = np.array([[o.f(ai) for o in objs] for ai in a])
    u = np.array([[o.u(ai) for o in objs] for ai in a])
    return f, u


def fu_pytmd(t, names, corrections="OTIS"):
    ts = timescale.from_calendar(t.year, t.month, t.day, hour=t.hour)
    pu, pf, _ = pyTMD.constituents.arguments(
        ts.MJD, names, corrections=corrections, deltat=ts.tt_ut1
    )
    return pf, np.rad2deg(pu)


def nodal_dataset(t, cons=CONS):
    """(f, u) for every constituent x package, as one tidy xr.Dataset."""
    utn, objs, tmn = zip(*cons.values())
    res = {
        "UTide":   fu_utide(t, utn),
        "pytides": fu_pytides(t, objs),
        "pyTMD":   fu_pytmd(t, tmn),
    }
    dims = ("time", "constituent")
    coords = {"time": t, "constituent": list(cons)}
    return xr.Dataset(
        {
            "f": (("package", *dims), np.stack([v[0] for v in res.values()])),
            "u": (("package", *dims), np.stack([wrap180(v[1]) for v in res.values()])),
        },
        coords={"package": list(res), **coords},
    )

t = pd.date_range("2010-01-01", "2029-01-01", freq="10D") # one tidal epoch
ds = nodal_dataset(t)
ds

Sorting by the worst spread tells us where to look. Three groups emerge:

- **Mf, Mm, MSf** — a large, constant disagreement. UTide reports a flat
  $f = 1$ here while the others vary by up to 30%. We confirm in Part 2 that
  this is a *missing correction*, not a different one.
- **M1, L2** — convention clashes (where $p$ sits relative to the astronomical
  argument), plus an `arctan` quadrant bug in pytides.
- **S1, R2, 2N2, S2, P1** — small residual differences in the satellite tables.

In [ ]:
spread = (ds.f.max("package") - ds.f.min("package")).max("time").to_series()
print(spread.sort_values(ascending=False).round(4).to_string())

### The figures

`f` and `u` against time, one panel per constituent, three packages overlaid,
ordered worst-disagreement first.

Two plotting traps here, both of which fail *silently*:

1. `groupby=...` + `.layout()` sorts facets **alphabetically**, discarding the
   worst-first ordering → build an explicit `hv.Layout`.
2. Panels share one y-range by default, flattening the small-amplitude
   constituents into featureless lines. `axiswise=True` unshares them, but it
   is an *element-level* option: it only takes effect on `hv.opts.Curve`, and is
   ignored on `hv.opts.NdOverlay`.

In [ ]:
LABEL = {"f": "Nodal factor $f$", "u": "Nodal phase $u$ [deg]"}
COLORS = {"UTide": "#d62728", "pytides": "#1f77b4", "pyTMD": "#2ca02c"}                                                                                                                                       
DASH   = {"UTide": "solid", "pytides": "dotted", "pyTMD": "dashed"}                                                                                                                                                        

def hv_panels(ds, names, var="f", ncols=4, width=400, height=300):                                                                                                                                            
    def overlay(c):                                                                                                                                                                                               
       return hv.NdOverlay(                                                                                                                                                                                      
           {                                                                                                                                                                                                     
               p: ds[var].sel(constituent=c, package=p)                                                                                                                                                          
                  .hvplot.line(x="time", dynamic=False)                                                                                                                                                          
                  .opts(color=COLORS[p], line_dash=DASH[p])                                                                                                                                             
               for p in ds.package.values                                                                                                                                                                        
           },                                                                                                                                                                                                    
           kdims="package",                                                                                                                                                                                      
       ).opts(title=c, ylabel=LABEL[var].replace("$", ""), xlabel="")                                                                                                                                                                                               
    return (                                                                                                                                                                                                  
        hv.Layout([overlay(c) for c in names])                                                                                                                                                                
        .cols(ncols)                                                                                                                                                                                          
        .opts(                                                                                                                                                                                                
            hv.opts.Curve(width=width, height=height, line_width=1.6,                                                                                                                                         
                            tools=["hover"], show_grid=True, shared_axes=False),                                                                                                                                   
            hv.opts.NdOverlay(legend_position="top_left", legend_cols=1, shared_axes=False),                                                                                                                                     
        )                                                                                                                                                                                                     
    )

order = spread.sort_values(ascending=False).index.tolist()
hv_panels(ds, order, "f").opts(title=f"Nodal factor over the 18.61-yr cycle  (lat {LAT}°N)")

In [ ]:
hv_panels(ds, order, "u").opts(title=f"Nodal phase over the 18.61-yr cycle  (lat {LAT}°N)")

---

## Part 2 - effects on a real record

Part 1 compared the tables. This part fits six years of cleaned IOC sea level
with both packages and asks where the difference actually lands.

Five stations, spanning latitude and tidal regime. `F = (K1+O1)/(M2+S2)` is the
form factor; range is the mean spring range `2(M2+S2+K1+O1)`:

| station | location | lat | range | F | regime | why this one |
|---|---|---|---:|---:|---|---|
| `trst` | Torres Strait, AU | −10.6° | 3.1 m | 1.19 | mixed, semidiurnal | near-equatorial, **largest S1** (29 mm) |
| `ohig3` | O'Higgins, Antarctica | −63.3° | 3.0 m | 0.90 | mixed, semidiurnal | high southern latitude, **large Mf** (27 mm) |
| `stma` | Saint-Malo, FR | +48.6° | 10.4 m | 0.04 | semidiurnal | macrotidal extreme, M2 = 3.6 m |
| `yaku` | Yakutat, AK | +59.5° | 4.4 m | 0.50 | mixed, semidiurnal | **largest Mf** (30 mm) |
| `adak` | Adak, AK | +51.9° | 1.8 m | 3.34 | **diurnal** | the only diurnal site; tests the K1/O1 corrections |

The two constituents that end up mattering, S1 and Mf, are not the dominant
ones anywhere — which is the point: the packages agree closely on everything
large.

To make the comparison attributable to nodal corrections, four things are held
equal — each of these produced a false signal larger than the real effect while
this was being written:

- **same constituent basis** — pytides' NOAA set includes `lambda2`, `M1`,
  `2MK3`, absent from Foreman's table; fit the intersection.
- **no trend term** — `utide.solve` defaults to `trend=True`, pytides has none.
- **centre-stamped hourly resampling** — an edge-labelled block biases every
  phase by 30 min of rotation (~0.25° at M2). Hourly is also a practical
  necessity: on raw 1-min data (3.1M points) `utide.solve` tries to allocate a
  98 GiB array in its confidence-interval step.
- **aligned $V_0$** — Greenwich lags are referenced to the equilibrium
  argument, and the two packages disagree on it by 76.7° for Sa and 167° for
  S1. Unaligned, Sa alone reports a 123 mm difference that is pure convention.

In [ ]:
import searvey
import ioc_cleanup as C
import panel as pn
import utide
from pytides2.tide import Tide

START, END = "2020-01-01", "2026-01-01"

# pytides' NOAA set minus the constituents with no entry in Foreman's table
COMMON = [c for c in pytidesc.noaa if c.name.upper() in UT_NAMES]
NAMES = [c.name.upper() for c in COMMON]
print(f"{len(NAMES)} common constituents; dropped "
      f"{[c.name for c in pytidesc.noaa if c.name.upper() not in UT_NAMES]}")
STATIONS = C.get_meta()

In [ ]:
def load(station: str, sensor: str) -> tuple[pd.Series, float, str]:
    """Cleaned, centre-stamped hourly series + latitude + label."""
    raw = searvey.fetch_ioc_station(station, START, END)
    ts = C.clean(raw, station, sensor).dropna()
    hourly = ts.resample("60min").mean().shift(freq="30min").dropna()
    row = STATIONS[STATIONS.ioc_code == station].iloc[0]
    lat, label = float(row.lat), row.location
    print(f"{station}/{sensor} {label}: lat={lat:.2f}  n={len(hourly):,}  "
          f"{hourly.index[0]:%Y-%m} to {hourly.index[-1]:%Y-%m}")
    return hourly, lat, label


def fit_both(ts: pd.Series, lat: float) -> pd.DataFrame:
    """Amplitude (m) and Greenwich phase (deg) from each package, V0 aligned.

    pytides' phase is shifted onto UTide's V0 convention so the two are
    comparable; see `dV0` below for why that is not optional.
    """
    even = ts.iloc[:-1] if len(ts) % 2 else ts  # pytides off-by-one on odd n
    tide = Tide.decompose(even.values, even.index.to_pydatetime(),
                          constituents=COMMON)[0]
    py_names = [c.name.upper() for c in tide.model["constituent"]]

    coef = utide.solve(ts.index, ts.values, lat=lat, constit=NAMES,
                       method="ols",      # pytides also does plain least squares
                       trend=False,       # pytides has no trend term
                       conf_int="none", phase="Greenwich", verbose=False)
    ut_names = [n.strip() for n in coef.name]

    out = pd.DataFrame({
        "A_py": pd.Series(tide.model["amplitude"], index=py_names),
        "g_py": pd.Series(tide.model["phase"] % 360, index=py_names),
        "A_ut": pd.Series(coef.A, index=ut_names),
        "g_ut": pd.Series(coef.g % 360, index=ut_names),
    }).dropna()

    out["g_ut"] -= dV0(lat, out.index)          # onto pytides' convention
    out["dA_mm"] = (out.A_ut - out.A_py) * 1000
    out["dg_deg"] = wrap180(out.g_ut - out.g_py)
    # vector difference
    out["vec_mm"] = (out.A_py * np.exp(1j * np.deg2rad(out.g_py)) - out.A_ut * np.exp(1j * np.deg2rad(out.g_ut))).abs() * 1000
    return out.sort_values("vec_mm", ascending=False)


def dV0(lat: float, names, t0=START) -> pd.Series:
    """UTide minus pytides equilibrium argument V0 [deg], per constituent."""
    tn = _normalize_time(np.array([pd.Timestamp(t0).to_datetime64()])
                         .astype("datetime64[s]"))
    a0 = py_astro(pd.Timestamp(t0).to_pydatetime())
    by_name = {c.name.upper(): c for c in COMMON}
    out = {}
    for n in names:
        if n in UT_NAMES and n in by_name:
            _, _, V = FUV(tn, tn[0], np.array([UT_NAMES.index(n)]), lat, [0, 0, 0, 0])
            out[n] = wrap180(V[0, 0] * 360 - by_name[n].V(a0)).item()
    return pd.Series(out).reindex(names).fillna(0.0)

In [ ]:
# --- Run all stations --------------------------------------------------------
STATIONS_TO_FIT = [("trst", "aqu"), ("ohig3", "prs"), ("stma", "rad"),
                   ("yaku", "pwl"), ("adak", "wls")]

results, STATION_SERIES = {}, {}
for station, sensor in STATIONS_TO_FIT:
    ts, lat, label = load(station, sensor)
    STATION_SERIES[station] = (ts, lat)        # reused in Part 3
    results[station] = (fit_both(ts, lat), lat, label)

In [ ]:
for station, (tab, lat, label) in results.items():
    print(f"\n=== {station} ({label}, lat {lat:.1f}) — "
        f"5 largest disagreements, of {len(tab)} constituents")
    print(tab[["A_py", "dA_mm", "dg_deg", "vec_mm"]].head(5)
        .assign(A_py=lambda d: d.A_py * 1000)
        .rename(columns={"A_py": "A_py_mm"}).round(2).to_string())

### Main differences on `S1` and `Mf`

`vec_mm` is the vector amplitude difference
$|A_{py}e^{ig_{py}} - A_{ut}e^{ig_{ut}}|$ — amplitude and phase folded into the
single quantity that drives the detided residual.

Everything outside the top few entries sits well below a millimetre: on the
major constituents (M2, S2, K1, O1, N2, K2) the two packages agree to
**~0.1–0.7 mm** on amplitudes of 100–560 mm. The disagreement concentrates in
**S1** at the equatorial station and **Mf** at the Antarctic one — and the two
have completely different causes.

### Per-constituent, both packages side by side


In [ ]:
# --- Bar charts: both packages per constituent, and the difference -----------
import hvplot.pandas  # noqa: F401 - registers the .hvplot accessor

COLORS = {"pytides": "#000000", "UTide": "#d62728"}
BANDS = {
    "Long-period":  ["SA", "SSA", "MM", "MSF", "MF"],
    "Diurnal":      ["2Q1", "Q1", "RHO1", "O1", "P1", "S1", "K1", "J1", "OO1"],
    "Semi-diurnal": ["2N2", "MU2", "N2", "NU2", "M2", "L2", "T2", "S2", "R2", "K2", "2SM2"],
    "Overtides":    ["M3", "MK3", "MN4", "M4", "MS4", "S4", "M6", "S6", "M8"],
}
BAND = {c: b for b, cs in BANDS.items() for c in cs}


def _tidy(tab: pd.DataFrame, var: str, relative: bool) -> pd.DataFrame:
    """Long-form frame for hvplot, with constituent and series order pinned.

    `source` must be an ordered Categorical: hvplot binds `color=[...]`
    positionally, so leaving it to incidental frame order is how the
    series/colour mapping silently swaps while the legend still reads right.
    """
    series = {
        ("amplitude", False): {"pytides": tab.A_py * 1000, "UTide": tab.A_ut * 1000},
        ("phase", False):     {"pytides": tab.g_py, "UTide": tab.g_ut},
        ("amplitude", True):  {"UTide \u2212 pytides": tab.dA_mm},
        ("phase", True):      {"UTide \u2212 pytides": tab.dg_deg},
    }[var, relative]

    cons = [c for band in BANDS.values() for c in band if c in tab.index]
    df = pd.DataFrame(series).loc[cons].set_axis(
        [f"{BAND[c]}\n{c}" for c in cons]
    ).rename_axis("label")

    out = df.reset_index().melt("label", var_name="source", value_name="value")
    out["label"] = pd.Categorical(out.label, df.index[::-1], ordered=True)   # first at top
    out["source"] = pd.Categorical(out.source, df.columns, ordered=True)
    return out.sort_values(["source", "label"])


def bars(tab: pd.DataFrame, var: str, relative: bool = False):
    unit = "mm" if var == "amplitude" else "deg"
    label = f"\u0394 {var} [{unit}]" if relative else f"{var} [{unit}]"
    return _tidy(tab, var, relative).hvplot.barh(
        x="label", y="value", by="source",
        xlabel="", ylabel=label, title=label,
        color=[COLORS["UTide"]] if relative else list(COLORS.values()),
        grid=True, legend="bottom_right",
    ).opts(
        width=360, height=900, bar_width=0.8, line_color=None,
        shared_axes=False, default_tools=["pan"],
        fontsize={"title": 11, "labels": 10, "xticks": 7, "yticks": 7},
    )


pn.Tabs(*(
    (station, pn.Row(bars(tab, "amplitude"), bars(tab, "phase"),
                     bars(tab, "amplitude", relative=True),
                     bars(tab, "phase", relative=True)))
    for station, (tab, lat, label) in results.items()
))

In [ ]:
# --- The two mechanisms, side by side ---------------------------------------
t = pd.date_range(START, END, freq="10D")
tn = _normalize_time(t.values.astype("datetime64[s]"))
astros = [py_astro(x.to_pydatetime()) for x in t]

rows = []
for name, obj in [("MF", pytidesc._Mf), ("S1", pytidesc._S1), ("MM", pytidesc._Mm)]:
    i = UT_NAMES.index(name)
    f_ut = FUV(tn, tn[0], np.array([i]), -63.32, [0, 0, 0, 0])[0][:, 0]
    f_py = np.array([obj.f(a) for a in astros])
    rows.append({
        "cons": name,
        "f_UTide": f"{f_ut.min():.3f}-{f_ut.max():.3f}",
        "f_pytides": f"{f_py.min():.3f}-{f_py.max():.3f}",
        "in Foreman table": "satellites" if not np.isnan(ut_constants.const.nsat[i])
                            else "NO ENTRY -> f=1, u=0",
    })
print(pd.DataFrame(rows).set_index("cons").to_string())

print("\nIf one package holds f wrong, least squares absorbs it into A.")
print("Predicted vs measured A_UTide / A_pytides:\n")
# S1's satellites have ilatfac=0, so its f is latitude-independent
f_s1 = FUV(tn, tn[0], np.array([UT_NAMES.index("S1")]), 45.0, [0, 0, 0, 0])[0][:, 0].mean()
f_mf = np.array([pytidesc._Mf.f(a) for a in astros]).mean()
for station, (tab, lat, label) in results.items():
    for name, predicted, why in [
        ("S1", 1 / f_s1, f"1/f_UTide = 1/{f_s1:.3f}"),
        ("MF", f_mf, f"mean f_pytides = {f_mf:.3f}"),
    ]:
        if name in tab.index:
            r = tab.loc[name]
            print(f"  {station:6s} {name}: measured {r.A_ut / r.A_py:.3f}  "
                  f"predicted {predicted:.3f}   ({why})")

### Mf — UTide applies no nodal correction at all

Mf, Mm, MSf, Sa, Ssa and T2 have **neither satellite entries nor a
shallow-water definition** in Foreman's table, so `FUV` returns exactly
$f=1$, $u=0$. pytides meanwhile swings Mf's $f$ between 0.99 and 1.45 over the
nodal cycle. These are the "FLAT" rows from Part 1, and the consequence is
visible in the fit: least squares has no choice but to absorb the missing
factor into the amplitude, so
$A_{\text{UTide}}/A_{\text{pytides}} \approx \overline{f_{\text{pytides}}} = 1.31$.

At O'Higgins, where Mf reaches 27 mm, that is an **11 mm amplitude error** —
the largest single discrepancy in the whole comparison.

### S1 — the opposite problem

Here it is pytides that uses $f = 1$ exactly, while UTide applies $f \approx 0.69$
from two satellite terms (`amprat` 0.353 and 0.026 — unusually large).
The fitted amplitudes differ by the reciprocal, $1/0.689 = 1.45$.

Neither is obviously "right": S1 in a tide gauge is largely **radiational** —
driven by the diurnal solar heating cycle, not the gravitational potential — so
a nodal factor derived from lunar node precession is questionable for it either
way. At Torres Strait, where S1 reaches 29 mm, this is a 13 mm difference.

> **Practical upshot.** For ordinary de-tiding both packages are equivalent to
> well under a millimetre on everything that matters. Switch to caring if your
> work touches the **fortnightly/monthly band** (don't trust UTide's $f$ for
> Mf/Mm/MSf) or **S1** (check which convention you want, and prefer fitting Sa
> and S1 from the data over any tabulated correction).

---

## Part 3 — in-matrix vs linearized times

The comparison above passes `[0, 0, 0, 0]` to `FUV`, i.e. UTide's **in-matrix**
method. Is that the fair comparison, and how much does the choice matter over
six years?

### What the two methods are

Harmonic analysis fits amplitudes and phases that are *constant*, while the
nodal factors $f_k$ and $u_k$ genuinely *vary* over the record. So you must
choose when to evaluate them. In UTide the entire difference is two lines of
[`harmonics.py`](https://github.com/wesleybowman/UTide/blob/master/utide/harmonics.py#L113-L116):

```python
if ngflgs[0]:                 # NodSatLint
    tt = np.array([tref])     # evaluate f, u ONCE at the reference time
else:                         # in-matrix (default)
    tt = t                    # evaluate f, u at EVERY timestamp
```

- **Linearized times** freezes $f_k, u_k$ at one reference time, so each design
  matrix column is a pure sinusoid with a constant envelope. $f_k(t_{ref})$ is
  then just a scalar multiplying the column — indistinguishable from the
  amplitude being solved for.
- **In-matrix** (Foreman et al. 2009) puts the time-varying $f_k(t), u_k(t)$
  *inside* the column, so the basis function is a sinusoid whose envelope swells
  and shrinks with the nodal cycle. The correction lives in the matrix rather
  than being applied to the answer afterwards.

$$
\begin{aligned}
\text{linearized:}&\quad E_{ik} = f_k(t_{ref})\,e^{2\pi i\,(U_k(t_{ref}) + V_k(t_i))}\\
\text{in-matrix:}&\quad E_{ik} = f_k(t_i)\,e^{2\pi i\,(U_k(t_i) + V_k(t_i))}
\end{aligned}
$$

### Is the comparison fair?

Yes. **pytides** partitions the record into 240-hour blocks and evaluates
$(f, u)$ at each block midpoint *inside* its least-squares residual
([`tide.py`](https://github.com/sahitono/pytides/blob/master/pytides2/tide.py#L440-L466)),
and **pyTMD** evaluates at every timestamp handed to it. All three are
in-matrix — pytides' version being a coarser, piecewise-constant approximation —
so `[0, 0, 0, 0]` is the like-for-like choice. Using `NodSatLint` would have
introduced an artefact of UTide's own, as the next cell shows.

In [ ]:
# --- Why in-matrix wins: test on synthetic data with a KNOWN answer -----------
# Build a 6-year record the honest way (f, u varying at every step), then try to
# recover the amplitudes we put in.
CHECK = {"M2": (1.000, 60.0), "K1": (0.400, 200.0),    # name: (amplitude m, phase deg)
         "O1": (0.250, 150.0), "N2": (0.200, 40.0)}

t_syn = pd.date_range(START, END, freq="1h")
tn = _normalize_time(t_syn.values.astype("datetime64[s]"))
lind = np.array([UT_NAMES.index(c) for c in CHECK])
tref = 0.5 * (tn[0] + tn[-1])        # UTide's own choice: the midpoint

F, U, V = FUV(tn, tref, lind, LAT, [0, 0, 0, 0])
A_true = np.array([a for a, _ in CHECK.values()])
g_true = np.deg2rad([g for _, g in CHECK.values()])
synthetic = pd.Series((F * A_true * np.cos(2 * np.pi * (U + V) - g_true)).sum(axis=1),
                      index=t_syn)

rows = []
for method, kw in [("in-matrix", {}), ("linearized", {"nodal": "linear_time"})]:
    co = utide.solve(synthetic.index, synthetic.values, lat=LAT, constit=list(CHECK),
                     method="ols", trend=False, conf_int="none",
                     phase="Greenwich", verbose=False, **kw)
    A = pd.Series(co.A, index=[n.strip() for n in co.name])
    g = pd.Series(co.g % 360, index=[n.strip() for n in co.name])
    for c, (a0, g0) in CHECK.items():
        rows.append({"method": method, "cons": c,
                     "dA_mm": (A[c] - a0) * 1000,
                     "dg_deg": wrap180(g[c] - g0).item()})

err = pd.DataFrame(rows).pivot(index="cons", columns="method")
print("Error against the known truth (6-yr synthetic record):\n")
print(err.round(3).to_string())

In-matrix recovers the input to machine precision — the model it fits *is* the
model that generated the data. Linearized times cannot, and the reason is that
$f_k(t_{ref}) \neq \langle f_k \rangle$: you are sampling a 6-year arc of an
18.6-year modulation, so the midpoint value is not the mean of the arc.

The data carry $f_k(t)\,A_k$, while the linearized model fits a constant
$f_k(t_{ref})$, so least squares returns
$A_{\text{fit}} \approx A_k\,\langle f_k\rangle / f_k(t_{ref})$ — the gap lands
directly in the amplitude.

In [ ]:
# --- The mechanism: the frozen f is not the mean f ---------------------------
# Data carry f(t)*A_true; the linearized model fits with a constant f(tref), so
# least squares returns  A_fit ~ A_true * <f> / f(tref).
rows = []
for c, (a0, _) in CHECK.items():
    li = np.array([UT_NAMES.index(c)])
    f_vary = FUV(tn, tref, li, LAT, [0, 0, 0, 0])[0][:, 0]   # f at every step
    f_froz = FUV(tn, tref, li, LAT, [1, 0, 0, 0])[0][0, 0]   # f frozen at tref
    rows.append({"cons": c, "f_min": f_vary.min(), "f_max": f_vary.max(),
                 "<f>": f_vary.mean(), "f(tref)": f_froz,
                 "<f>/f(tref)": f_vary.mean() / f_froz,
                 "predicted dA_mm": (f_vary.mean() / f_froz - 1) * a0 * 1000})
gap = pd.DataFrame(rows).set_index("cons")
gap["actual dA_mm"] = err[("dA_mm", "linearized")]
print(gap.round(4).to_string())

Predicted and actual agree, so the mechanism is understood. The consequence is
that the linearization error **grows with record length**, which is exactly the
reason the old guidance capped nodal corrections at 1–2 years. On real data:

In [ ]:
# --- Same test on the real records, and vs record length ---------------------
def linearization_error(ts: pd.Series, lat: float) -> pd.Series:
    """Amplitude shift [mm] from using linearized times instead of in-matrix."""
    fit = lambda **kw: pd.Series(
        (co := utide.solve(ts.index, ts.values, lat=lat, constit=NAMES, method="ols",
                           trend=False, conf_int="none", phase="Greenwich",
                           verbose=False, **kw)).A,
        index=[n.strip() for n in co.name])
    return (fit(nodal="linear_time") - fit()) * 1000


print("Largest amplitude shifts from linearized times, full records [mm]:\n")
for station, (_, lat, label) in results.items():
    d = linearization_error(*STATION_SERIES[station])
    worst = d.abs().nlargest(4)
    print(f"  {station:6s} RMS {np.sqrt((d ** 2).mean()):5.2f} mm | "
          + "  ".join(f"{c} {d[c]:+6.2f}" for c in worst.index))

print("\nSame station (trst), shrinking the window — the error scales with span:\n")
ts, lat = STATION_SERIES["trst"]
for label, seg in [("6 yr", ts), ("2 yr", ts.loc["2020":"2021"]), ("1 yr", ts.loc["2020"])]:
    d = linearization_error(seg, lat)
    print(f"  {label}: max |dA| {d.abs().max():5.2f} mm, RMS {np.sqrt((d ** 2).mean()):4.2f} mm"
          f"   (O1 {d['O1']:+5.2f}, K1 {d['K1']:+5.2f})")

O1 and K1 degrade from a few hundredths of a millimetre at one year to ~8 mm at
six — **larger than the UTide-vs-pytides difference** for those constituents.
Two things follow:

1. The in-matrix default is doing real work, and the old 1–2 year restriction on
   nodal corrections was well founded for the linearized method.
2. `tref` matters. UTide uses the midpoint,
   [`_solve.py#L522`](https://github.com/wesleybowman/UTide/blob/master/utide/_solve.py#L522):
   `tref = 0.5 * (tin[0] + tin[-1])`. That centres the arc and keeps
   $\langle f\rangle - f(t_{ref})$ as small as it can be; referencing to the
   series start would roughly double the lever arm.

So for the question that started this: the differences between the three
packages' nodal factors are real but, for de-tiding, smaller than the choice of
*method* within a single package once the record gets long.